In [1]:
import pandas as pd

In [2]:
df_uo = pd.read_csv('../data/raw/etl_users_orders_dirty_v3.csv')

In [3]:
#Первым делом декомпозируем исходную таблицу и получаем два дата фрейма:
df_u = df_uo[['user_id', 'name', 'city', 'age', 'source', 'registered_at']]
df_o = df_uo[['user_id', 'order_id', 'order_date', 'category', 'amount', 'status']]

In [4]:
#Удаляем дубликаты user_id в новом датафрейме df_u, т.к. он становится первичным ключом
df_u = df_u.drop_duplicates(subset=['user_id'])
df_u = df_u.sort_values('user_id')

In [5]:
#Перевел колонку с регистрацией из типа данных str в datetime 64
df_u['registered_at'] = pd.to_datetime(df_u['registered_at'], errors='coerce')

In [6]:
#Сначала подготовил колонку с возрастом и перевел все неподходящие под числовый тип данные в NaN
#Перевел в тип данных Int64 - nullable integer для хранения NaN
df_u['age'] = df_u['age'].replace('unknown', pd.NA)
df_u['age'] = df_u['age'].astype('Int64')

In [7]:
#Явно видны выбросы по возрасту: 0, 5 и 150, заменим их на Nan
df_u['age'] = df_u['age'].replace(0, pd.NA).replace(150, pd.NA).replace(5, pd.NA)

In [8]:
#Теперь явно видно: средний возраст 41, минимальный - 18, максимальный 65
df_u['age'].describe()

count      39897.0
mean     41.472266
std      13.868967
min           18.0
25%           29.0
50%           41.0
75%           54.0
max           65.0
Name: age, dtype: Float64

In [9]:
#В строковых типах данных для наглядности заменим пустые значения на "unknown"
df_u['name'] = df_u['name'].fillna('unknown')
df_u['city'] = df_u['city'].fillna('unknown')
df_u['source'] = df_u['source'].fillna('unknown')

In [10]:
#"Подчистим" названия городов
df_u['city'] = df_u['city'].str.capitalize().str.strip()
df_u['city'] = df_u['city'].replace('Спб', 'Санкт-петербург')
df_u['city'] = df_u['city'].replace('Санкт петербург', 'Санкт-петербург')
df_u['city'] = df_u['city'].replace('Санкт-петербург', 'Санкт-Петербург')

In [11]:
#Даты корректны
df_u['registered_at'].describe()

count                         39909
mean     2025-09-07 11:12:04.438096
min             2025-01-01 00:00:00
25%             2025-05-04 15:00:00
50%             2025-09-07 04:00:00
75%             2026-01-11 01:00:00
max             2026-05-16 23:00:00
Name: registered_at, dtype: object

In [12]:
#Удаляем дубликаты order_id в новом датафрейме df_o, т.к. он становится первичным ключом
df_o = df_o.drop_duplicates(subset='order_id')

In [13]:
#Перевел колонку с регистрацией из типа данных str в datetime 64
df_o['order_date'] = pd.to_datetime(df_o['order_date'], errors='coerce')

In [14]:
#В строковых типах данных для наглядности заменим пустые значения на "unknown"
df_o['status'] = df_o['status'].fillna('unknown')

In [15]:
#Работаем с amount
df_o['amount'] = pd.to_numeric(df_o['amount'], errors='coerce')

In [16]:
#Избавляемся от отрицательных сумм
df_o = df_o[(df_o['amount'] > 0) | (df_o['amount'].isnull())]

In [17]:
#Финальные проверки:
df_u['user_id'].duplicated().sum()

np.int64(0)

In [18]:
df_o['order_id'].duplicated().sum()

np.int64(0)

In [19]:
df_o['user_id'].duplicated().sum()

np.int64(80764)

In [20]:
df_o['user_id'].isin(df_u['user_id']).all()

np.True_

In [21]:
#Сохраним очищенные данные:
df_u.to_csv('../data/processed/users_clean.csv', index=False)

In [22]:
df_o.to_csv('../data/processed/orders_clean.csv', index=False)